In [ ]:
#!/usr/bin/env python3
"""
Python script to automatically generate MATLAB scripts for NEVIS ice dynamics simulations
Author: Auto-generated script
Date: 2025-11
"""

import os
import glob
import argparse
import sys
import numpy as np
from pathlib import Path

def format_scientific(value):
    """Format scientific notation for filenames"""
    if value >= 1:
        sci_str = f"{value:.0e}".replace('e+0', 'e').replace('e+', 'e')
    else:
        sci_str = f"{value:.0e}"
    return sci_str.replace('-', '_').replace('+', '')

def generate_ice_spinup_script(moulin_input=0, eps=0.1, kappa=1e-10, mu=1e2, Nx=41, Ny=41, output_dir="./"):
    """
    Generate MATLAB script for ice dynamics spinup
    
    Parameters:
    -----------
    eps : ratio of membrane stress terms in momentum equation (default: 0.1)
    kappa : leakage coefficient (default: 1e-10)
    mu : water viscosity in Pa·s (default: 1e2)
    Nx : number of grid points in x direction (default: 41)
    Ny : number of grid points in y direction (default: 21)
    output_dir : directory to save the generated script (default: current directory)
    """
    
    # Format parameters for filename
    moulin_input_str = format_scientific(moulin_input)
    eps_str = format_scientific(eps)
    kappa_str = format_scientific(kappa)
    mu_str = format_scientific(mu)
    casename = f"n2d_moulin{moulin_input_str}_eps{eps_str}_kappa{kappa_str}_mu{mu_str}_spinup"
    matlab_script = f"""% Ice dynamics spinup script
% Generated automatically for parameter sweep
% Date: 2025-11

clear 
oo.root = './';             % filename root
oo.code = '../nevis/src';   % code directory
oo.results = 'results';     % path to the results folders
addpath(oo.code);           % add path to code
oo.casename = '{casename}';
oo.fn = ['/',oo.casename];               % filename (same as casename)
oo.rn = [oo.root,oo.results,oo.fn];      % path to the case results
mkdir(oo.rn);

%% parameters
[pd,oo] = nevis_defaults([],oo);
oo.evaluate_variables = 1;
oo.input_gaussian = 1;
oo.relaxation_term = 1;                         % 1: proportional to pressure diff
oo.initial_condition = 0;                       
oo.cavity_coupling = 1;                         % couple to ice velocity
oo.display_residual = 0;

moulin_input = {moulin_input};                  % prescribed moulin input (m^3/s)
pd.mu = {mu};                                   % water viscosity (Pa s)
pd.Ye = 8.8e9;                                  % Young's modulus (Pa)
pd.B = pd.Ye*(1e3)^3/(12*(1-0.33^2));           % bending stiffness (Pa m^3)
pd.E_lapse = 30/1000/pd.td/10^3;

pd.k_s = 1e-3;                                  % sheet roughness parameter

pd.hb_reg1 = 5e-3;                              % Regularisation parameter for hb
pd.hb_reg2 = 1e-3;                              % Regularisation parameter for hb
pd.N_reg1 = 1e4;                                % Regularisation parameter for N
pd.deltap_reg = 1e4;                            % Regularisation parameter for deltap
pd.B_reg = 0;                                   % Regularisation parameter for B

pd.G = 0.01;                                    % geothermal heat flux [J/s/m^2]
pd.melt = pd.G/pd.rho_w/pd.L;                   % geothermal heat derived basal melt [m/s]
pd.alpha_b = 0;                                 % relaxation rate (s^-1)
pd.kappa_b = {kappa};                             % relaxation coeff
pd.c0 = 1;

[ps,pp] = nevis_nondimension(pd,[],oo);

%% grid and geometry
L = 100000;                                     % length in m
W = 50000;                                      % width in m
x = linspace(0,(L/ps.x),{Nx}); 
y = linspace(0,(W/ps.x),{Ny});
gg = nevis_grid(x,y,oo);
b = (0/ps.z)*gg.nx;
s = (1000/ps.z)*max(1-gg.nx/max(max(gg.nx)),0).^(1/2);

%% mask grid
gg = nevis_mask(gg,find(s-b<=0)); 
gg.n1m = gg.n1;                 % margin boundary nodes
gg = nevis_label(gg,gg.n1m);    % label pressure boundary nodes

%% plot grid
nevis_plot_grid(gg);

%% initialize
[aa,vv] = nevis_initialize(b,s,gg,pp,oo);
pd.k_f = 0.9;                                   % percent overburden (k-factor)
vv.phi = aa.phi_a+pd.k_f*(aa.phi_0-aa.phi_a);   % 90% overburden 
vv.hs = (0.1/ps.hs)*ones(gg.nIJ,1);             % 10cm thick sheet
vv.hb = (0.1/ps.hb)*ones(gg.nIJ,1);             % 10cm thick bed

%% boundary conditions
aa.phi_b = max(aa.phi_0,aa.phi_a);              % prescribed boundary pressure

%% add parameters and boundary labels for ice velocity
pd.n_glen = 1;
eps = {eps}; 
pd.A_glen = 1/2/((eps)*pd.rho_i*pd.g*ps.z*ps.x/pd.u_b);
[pd,ps,pp,oo] = nevis_add_parameters_ice(pd,ps,pp,oo);

gg = nevis_label_ice(gg);
figure(1); clf; 
nevis_plot_grid_ice(gg); 

%% initial ice velocity
N = ones(gg.nIJ,1); 
u = zeros(gg.eIJ,1); 
v = zeros(gg.fIJ,1); 

u(gg.eout2) = NaN; 
v(gg.fout2) = NaN;
[u,v] = nevis_velocity(aa.H,u,v,N,aa,pp,gg,oo);

vv.u = u; 
vv.v = v;

oo.include_ice = 1;

%% moulins 
oo.keep_all_moulins = 0;
oo.random_moulins = 0;         
[pp.ni_m,pp.sum_m] = nevis_moulins([0.05*L/ps.x],[0.5*W/ps.x],gg,oo);     % one moulin at the lake location         

%% supraglacial lakes
pp.x_l = [0.25*L/ps.x];
pp.y_l = [0.5*W/ps.x];
pp.V_l = [0e7/(ps.Q0*ps.t)];
pp.t_drainage = [0*365*pd.td/ps.t];
pp.t_duration = [0.025*pd.td/ps.t];
[pp.ni_l,pp.sum_l] = nevis_lakes(pp.x_l,pp.y_l,gg,oo);
oo.pts_ni = [pp.ni_l; pp.ni_m];  

%% surface input
oo.surface_runoff = 0;                          
oo.RACMO_runoff = 0;                            
oo.distributed_input = 0;                       
pp.meltE = @(t) (0/1000/pd.td/ps.m)*(1-exp(-t/(30*pd.td/ps.t)));
pp.input_function = @(t) moulin_input*(1-exp(-t/(30*pd.td/ps.t)))./(ps.m*ps.x^2);    

%% save initial parameters
save([oo.rn,oo.fn],'pp','pd','ps','gg','aa','vv','oo');

%% timestep 
oo.dt = 1/24*pd.td/ps.t; 
oo.save_timesteps = 1; 
oo.save_pts_all = 1; 
oo.t_span = (0:1:365*10)*pd.td/ps.t;         
[tt,vv] = nevis_timesteps_ice(oo.t_span,vv,aa,pp,gg,oo);

%% expand/update variables
aa = nevis_inputs(vv.t,aa,vv,pp,gg,oo);
oo.evaluate_variables = 1; 
[vv2] = nevis_backbone(inf,vv,vv,aa,pp,gg,oo); 
vv2 = nevis_nodedischarge(vv2,aa,pp,gg,oo); 
save([oo.rn,oo.fn],'pp','pd','ps','gg','aa','vv','oo','tt');
"""
    
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    output_file = os.path.join(output_dir, f"{casename}.m")
    
    with open(output_file, 'w') as f:
        f.write(matlab_script)
    
    print(f"  Generated: {casename}.m")
    return output_file

def generate_ice_drainage_script(moulin_input=0, eps=0.1, kappa=1e-10, mu=1e2, V=1e7, Nx=41, Ny=41, output_dir="./"):
    """Generate MATLAB script for ice dynamics drainage test"""
    
    # Format parameters
    moulin_input_str = format_scientific(moulin_input)
    eps_str = format_scientific(eps)
    kappa_str = format_scientific(kappa)
    mu_str = format_scientific(mu)
    log10V = int(round(np.log10(V)))

    spinup_base = f"n2d_moulin{moulin_input_str}_eps{eps_str}_kappa{kappa_str}_mu{mu_str}"
    casename = f"{spinup_base}_V1e{log10V}_drainage"
    initname = f"{spinup_base}_spinup"

    matlab_script = f"""% Ice dynamics drainage script
% Generated automatically for parameter sweep
% Date: 2025-11

clear
%% read in the initial condition
casename = '{casename}';
initname = '{initname}';

data = load(['./results/' initname '/' initname]);
pd = data.pd;
ps = data.ps;
pp = data.pp;
aa = data.aa;
oo = data.oo;

oo.casename = casename;
oo.initname = initname;

oo.fn = ['/',oo.casename];
oo.rn = [oo.root,oo.results,oo.fn];
addpath(oo.code);
mkdir(oo.rn);
oo.Tol_F = 1e-7;
oo.cavity_coupling = 0;                        % couple to ice velocity

%% grid and geometry
L = 100000;
W = 50000;
x = linspace(0,(L/ps.x),{Nx}); 
y = linspace(0,(W/ps.x),{Ny});
gg = nevis_grid(x,y,oo);
b = (0/ps.z)*gg.nx;
s = (1000/ps.z)*max(1-gg.nx/max(max(gg.nx)),0).^(1/2);

%% mask grid
gg = nevis_mask(gg,find(s-b<=0)); 
gg.n1m = gg.n1;
gg = nevis_label(gg,gg.n1m);

%% add ice velocity labels
gg = nevis_label_ice(gg);

%% initialize from spinup
init_cond = load(['./results/' oo.initname '/' '3650.mat']);
vv = init_cond.vv;

[pd,ps,pp,oo] = nevis_add_parameters_ice(pd,ps,pp,oo); % add parameters etc needed to solve for ice 

%% supraglacial lakes
pp.x_l = [0.25*L/ps.x];                                         % x-coord of lakes
pp.y_l = [0.5*W/ps.x];                                          % y-coord of lakes
pp.V_l = [1e{log10V}/(ps.Q0*ps.t)];                             % volume of lakes         
pp.t_drainage = vv.t + 0.2*365*pd.td/ps.t;                      % time of lake drainages
pp.t_duration = [0.025*pd.td/ps.t];                             % duration of lake drainages
[pp.ni_l,pp.sum_l] = nevis_lakes(pp.x_l,pp.y_l,gg,oo);          % calculate lake catchments 
oo.pts_ni = [pp.ni_l; pp.ni_m];  

%% save initial parameters
save([oo.rn,oo.fn],'pp','pd','ps','gg','aa','vv','oo');

%% timestep 
oo.dt = 1/24*pd.td/ps.t; 
oo.save_timesteps = 1; 
oo.save_pts_all = 1; 
oo.t_span = vv.t + (0:0.25:365*2.0)*pd.td/ps.t;         
[tt,vv] = nevis_timesteps_ice(oo.t_span,vv,aa,pp,gg,oo);     % save at daily timesteps

save([oo.rn,oo.fn],'pp','pd','ps','gg','aa','vv','oo','tt');
"""
    
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    output_file = os.path.join(output_dir, f"{casename}.m")
    
    with open(output_file, 'w') as f:
        f.write(matlab_script)
    
    print(f"  Generated: {casename}.m")
    return output_file

def batch_generate_ice_dynamics(parameter_combinations, Nx=41, Ny=21, if_spinup=True, if_drainage=True, output_dir="./generated_scripts/ice_dynamics/"):
    """
    Generate batch of ice dynamics scripts
    
    Parameters:
    -----------
    parameter_combinations : list of tuples (eps, kappa, mu, V)
    Nx, Ny : grid resolution
    if_spinup : generate spinup scripts
    if_drainage : generate drainage scripts
    output_dir : output directory
    """
    
    # Create output directories
    spinup_dir = os.path.join(output_dir, 'spinup')
    drainage_dir = os.path.join(output_dir, 'drainage')
    
    if if_spinup:
        Path(spinup_dir).mkdir(parents=True, exist_ok=True)
        # Clean old files
        old_files = glob.glob(os.path.join(spinup_dir, "*.m"))
        for f in old_files:
            os.remove(f)
    
    if if_drainage:
        Path(drainage_dir).mkdir(parents=True, exist_ok=True)
        # Clean old files
        old_files = glob.glob(os.path.join(drainage_dir, "*.m"))
        for f in old_files:
            os.remove(f)
    
    generated_files = []
    
    print(f"\nGenerating scripts with grid resolution: Nx={Nx}, Ny={Ny}")
    print("-" * 60)
    
    for i, (moulin_input, eps, kappa, mu, V) in enumerate(parameter_combinations, 1):
        print(f"\n[{i}/{len(parameter_combinations)}] eps={eps}, kappa={kappa:.0e}, mu={mu:.0e}, V={V:.0e}")
        
        if if_spinup:
            output_file = generate_ice_spinup_script(
                moulin_input=moulin_input, eps=eps, kappa=kappa, mu=mu, Nx=Nx, Ny=Ny, output_dir=spinup_dir
            )
            generated_files.append(output_file)
        
        if if_drainage and V > 0:
            output_file = generate_ice_drainage_script(
                moulin_input=moulin_input, eps=eps, kappa=kappa, mu=mu, V=V, Nx=Nx, Ny=Ny, output_dir=drainage_dir
            )
            generated_files.append(output_file)
    
    return generated_files

def main():
    """Main function to generate MATLAB scripts"""
    
    # Check if running in Jupyter
    if 'ipykernel' in sys.modules:
        print("="*60)
        print("Running in Jupyter environment")
        print("="*60)
        
        # Switches
        if_spinup = True
        if_drainage = True
        case = 2  # Select case
        
        # Grid resolution
        Nx = 41
        Ny = 21
        
        # Define parameter combinations by case
        if case == 1:
            # Reference cases
            print("Case 1: Reference cases")
            parameter_combinations = [
                (0, 0.1, 1e-10, 1e1, 0),      # spinup only
                (0, 0.1, 1e-10, 1e1, 1e7),    # baseline drainage
            ]
        elif case == 2:
            # Vary eps
            print("Case 2: Varying eps")
            parameter_combinations = [
                (0, 0.01, 1e-10, 1e1, 1e7),
                (0, 0.1, 1e-10, 1e1, 1e7),
                (0, 1.0, 1e-10, 1e1, 1e7),
                (0, 10.0, 1e-10, 1e1, 1e7),
                # (0, 0.01, 1e-10, 1e0, 1e7),
                # (0, 0.1, 1e-10, 1e0, 1e7),
                # (0, 1.0, 1e-10, 1e0, 1e7),
                # (0, 10.0, 1e-10, 1e0, 1e7),
                (10, 0.01, 1e-10, 1e1, 1e7),
                (10, 0.1, 1e-10, 1e1, 1e7),
                (10, 1.0, 1e-10, 1e1, 1e7),
                (10, 10.0, 1e-10, 1e1, 1e7),
            ]
        elif case == 3:
            # Vary kappa
            print("Case 3: Varying kappa")
            parameter_combinations = [
                (0, 0.1, 1e-12, 1e2, 1e7),
                (0, 0.1, 1e-11, 1e2, 1e7),
                (0, 0.1, 1e-10, 1e2, 1e7),
                (0, 0.1, 1e-9, 1e2, 1e7),
                (0, 0.1, 1e-8, 1e2, 1e7),
            ]
        elif case == 4:
            # Vary mu
            print("Case 4: Varying mu")
            parameter_combinations = [
                (0, 0.1, 1e-10, 1e1, 1e7),
                (0, 0.1, 1e-10, 1e2, 1e7),
                (0, 0.1, 1e-10, 1e3, 1e7),
                (0, 0.1, 1e-10, 1e4, 1e7),
            ]
        elif case == 5:
            # Vary V
            print("Case 5: Varying lake volume")
            parameter_combinations = [
                (0, 0.1, 1e-10, 1e2, 0),      # no drainage
                (0, 0.1, 1e-10, 1e2, 1e6),
                (0, 0.1, 1e-10, 1e2, 1e7),
                (0, 0.1, 1e-10, 1e2, 1e8),
            ]
        else:
            print("Invalid case number")
            return
        
        output_dir = "./generated_scripts/ice_dynamics/"
        
        generated_files = batch_generate_ice_dynamics(
            parameter_combinations, 
            Nx=Nx, 
            Ny=Ny,
            if_spinup=if_spinup,
            if_drainage=if_drainage,
            output_dir=output_dir
        )
        
        print("\n" + "="*60)
        print(f"Successfully generated {len(generated_files)} scripts")
        print("="*60)
        print(f"\nSpinup scripts: {output_dir}spinup/")
        print(f"Drainage scripts: {output_dir}drainage/")
        
        return generated_files
    
    # Command line interface (same as before)
    # ... [keep existing CLI code]
    
if __name__ == "__main__":
    main()

# Jupyter quick start
if 'ipykernel' in sys.modules and __name__ != "__main__":
    print("="*60)
    print("Ice Dynamics Script Generator")
    print("="*60)
    print("\nUsage:")
    print("  generated_files = main()")
    print("\nEdit the 'case' variable in main() to select parameter combinations:")
    print("  case 1: Reference cases")
    print("  case 2: Varying eps")
    print("  case 3: Varying kappa")
    print("  case 4: Varying mu")
    print("  case 5: Varying lake volume")

Running in Jupyter environment
Case 2: Varying eps

Generating scripts with grid resolution: Nx=41, Ny=21
------------------------------------------------------------

[1/8] eps=0.01, kappa=1e-10, mu=1e+01, V=1e+07
  Generated: n2d_moulin0e00_eps1e_02_kappa1e_10_mu1e1_spinup.m
  Generated: n2d_moulin0e00_eps1e_02_kappa1e_10_mu1e1_V1e7_drainage.m

[2/8] eps=0.1, kappa=1e-10, mu=1e+01, V=1e+07
  Generated: n2d_moulin0e00_eps1e_01_kappa1e_10_mu1e1_spinup.m
  Generated: n2d_moulin0e00_eps1e_01_kappa1e_10_mu1e1_V1e7_drainage.m

[3/8] eps=1.0, kappa=1e-10, mu=1e+01, V=1e+07
  Generated: n2d_moulin0e00_eps1e0_kappa1e_10_mu1e1_spinup.m
  Generated: n2d_moulin0e00_eps1e0_kappa1e_10_mu1e1_V1e7_drainage.m

[4/8] eps=10.0, kappa=1e-10, mu=1e+01, V=1e+07
  Generated: n2d_moulin0e00_eps1e1_kappa1e_10_mu1e1_spinup.m
  Generated: n2d_moulin0e00_eps1e1_kappa1e_10_mu1e1_V1e7_drainage.m

[5/8] eps=0.01, kappa=1e-10, mu=1e+01, V=1e+07
  Generated: n2d_moulin1e1_eps1e_02_kappa1e_10_mu1e1_spinup.m
  Generat